# Synthetic B2B Dataset

This notebook creates a realistic relational dataset for a fictional international B2B company. It simulates dimensions and fact tables for revenue analysis, forecasting, profitability, inventory, CRM activity, returns and sales pipeline reporting.

The data is fully synthetic and does not contain confidential customer information.

## Notebook overview

We build the dataset in the following order:

1. **Configure the generator** — define the random seed, time period, dataset sizes and output folder.
2. **Create dimension tables** — dates, regions, products, customers and sales representatives.
3. **Generate sales transactions** — sample valid customer–product–date combinations and calculate units, prices, discounts, revenue, costs and margin.
4. **Create supporting fact tables** — forecasts, inventory, costs, CRM activities, returns and pipeline opportunities.
5. **Validate the relational model** — check schemas, primary keys, foreign keys, business rules and date consistency.
6. **Export the data** — save all tables as semicolon-separated CSV files for SQL Server and Power BI.

## Data model

| Table | Type | Grain |
|---|---|---|
| `dimDate` | Dimension | One row per calendar day |
| `dimRegion` | Dimension | One row per country/market |
| `dimProduct` | Dimension | One row per product/SKU |
| `dimCustomer` | Dimension | One row per customer |
| `dimSalesRep` | Dimension | One row per sales representative |
| `factSales` | Fact | One row per sales transaction |
| `factForecast` | Fact | One row per month, product, and region |
| `factInventory` | Fact | One row per month, product, and region |
| `factCosts` | Fact | One row per month and product |
| `factCRMActivities` | Fact | One row per customer interaction |
| `factReturns` | Fact | One row per returned sales transaction |
| `factPipeline` | Fact | One row per sales opportunity |

# 1. Setup 

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [3]:
# Reproducibility and time horizon
SEED = int(os.getenv("SYNTHETIC_DATA_SEED", "42"))
DATE_START = os.getenv("SYNTHETIC_DATA_START", "2021-01-01")
DATE_END = os.getenv("SYNTHETIC_DATA_END", "2025-12-31")

# Dataset sizes
N_SALES_ROWS = int(os.getenv("SYNTHETIC_N_SALES", "120000"))
N_CUSTOMERS = int(os.getenv("SYNTHETIC_N_CUSTOMERS", "1200"))
N_SALES_REPS = int(os.getenv("SYNTHETIC_N_SALES_REPS", "85"))
N_CRM_ACTIVITIES = int(os.getenv("SYNTHETIC_N_CRM", "35000"))
N_OPPORTUNITIES = int(os.getenv("SYNTHETIC_N_PIPELINE", "15000"))
PRODUCTS_PER_GROUP = int(os.getenv("SYNTHETIC_PRODUCTS_PER_GROUP", "20"))
RETURN_RATE = 0.035

OUTPUT_DIR = Path("../data/full_data/generated_data_tables")

rng = np.random.default_rng(SEED)

print(f"Date range:      {DATE_START} to {DATE_END}")
print(f"Sales rows:     {N_SALES_ROWS:,}")
print(f"Customers:      {N_CUSTOMERS:,}")
print(f"Output folder:  {OUTPUT_DIR.resolve()}")
print(f"Random seed:    {SEED}")


Date range:      2021-01-01 to 2025-12-31
Sales rows:     120,000
Customers:      1,200
Output folder:  C:\Benutzer\Anastasia\Lokale Daten\AS Portfolio\early-warning\data\full_data\generated_data_tables
Random seed:    42


# 2. Create dimension tables

In [33]:
dates = pd.date_range(DATE_START, DATE_END, freq="D")

dimDate = pd.DataFrame({"date": dates})
dimDate["date_id"] = dimDate["date"].dt.strftime("%Y%m%d").astype(int)
dimDate["year"] = dimDate["date"].dt.year
dimDate["quarter"] = dimDate["date"].dt.quarter
dimDate["month"] = dimDate["date"].dt.month
dimDate["month_name"] = dimDate["date"].dt.month_name()
dimDate["year_month"] = dimDate["date"].dt.to_period("M").astype(str)
dimDate["day_of_week"] = dimDate["date"].dt.dayofweek + 1
dimDate["week_of_year"] = dimDate["date"].dt.isocalendar().week.astype(int)
dimDate["is_month_end"] = dimDate["date"].dt.is_month_end
dimDate["is_quarter_end"] = dimDate["date"].dt.is_quarter_end
dimDate["date"] = dimDate["date"].dt.strftime("%Y-%m-%d")

date_series = pd.to_datetime(dimDate["date"])
date_array = date_series.to_numpy()

print(f"dimDate shape: {dimDate.shape}")
display(dimDate.head(3))
display(dimDate.tail(3))


dimDate shape: (1826, 11)


,date,date_id,year,quarter,month,month_name,year_month,day_of_week,week_of_year,is_month_end,is_quarter_end
0,2021-01-01,20210101,2021,1,1,January,2021-01,5,53,False,False
1,2021-01-02,20210102,2021,1,1,January,2021-01,6,53,False,False
2,2021-01-03,20210103,2021,1,1,January,2021-01,7,53,False,False


,date,date_id,year,quarter,month,month_name,year_month,day_of_week,week_of_year,is_month_end,is_quarter_end
1823,2025-12-29,20251229,2025,4,12,December,2025-12,1,1,False,False
1824,2025-12-30,20251230,2025,4,12,December,2025-12,2,1,False,False
1825,2025-12-31,20251231,2025,4,12,December,2025-12,3,1,True,True


## 2.2 Region dimension

Regions contain currency conversion, market-growth and margin assumptions. All core monetary measures are stored in EUR while `fx_to_eur` also allows local-currency reporting.

In [34]:
dimRegion = pd.DataFrame(
    [
        (1, "Germany", "Western EU", "EUR", 1.00, 1.06, 0.98),
        (2, "France", "Western EU", "EUR", 1.00, 1.02, 0.97),
        (3, "Spain", "Southern EU", "EUR", 1.00, 1.09, 0.94),
        (4, "Italy", "Southern EU", "EUR", 1.00, 1.01, 0.93),
        (5, "United Kingdom", "Northern EU", "GBP", 1.17, 0.97, 1.01),
        (6, "Ireland", "Northern EU", "EUR", 1.00, 1.07, 1.00),
        (7, "Poland", "Eastern EU", "PLN", 0.23, 1.12, 0.89),
        (8, "Slovakia", "Eastern EU", "EUR", 1.00, 1.05, 0.95),
        (9, "Belgium", "Western EU", "EUR", 1.00, 1.08, 0.96),
        (10, "Finland", "Northern EU", "EUR", 1.00, 1.04, 0.92),],
    columns=[
        "region_id",
        "country",
        "region",
        "currency",
        "fx_to_eur",
        "market_growth_factor",
        "margin_factor"])

print(f"dimRegion shape: {dimRegion.shape}")
display(dimRegion)

dimRegion shape: (10, 7)


,region_id,country,region,currency,fx_to_eur,market_growth_factor,margin_factor
0,1,Germany,Western EU,EUR,1.00,1.06,0.98
1,2,France,Western EU,EUR,1.00,1.02,0.97
2,3,Spain,Southern EU,EUR,1.00,1.09,0.94
3,4,Italy,Southern EU,EUR,1.00,1.01,0.93
4,5,United Kingdom,Northern EU,GBP,1.17,0.97,1.01
5,6,Ireland,Northern EU,EUR,1.00,1.07,1.00
6,7,Poland,Eastern EU,PLN,0.23,1.12,0.89
7,8,Slovakia,Eastern EU,EUR,1.00,1.05,0.95
8,9,Belgium,Western EU,EUR,1.00,1.08,0.96
9,10,Finland,Northern EU,EUR,1.00,1.04,0.92


## 2.3 Product dimension

Each product group contains several individual SKUs. Prices and costs vary around a group-level baseline. Lifecycle stages influence later sampling, pricing and sales behavior.

Instead of assigning lifecycle stages independently of launch year, we use product age as a guide. A recently launched product is therefore more likely to be `New` or `Growth`, while an older product is more likely to be `Mature` or `Decline`.

In [35]:
product_groups = [
    ("Laptop Pro", "IT", 950, 0.29, 1.08),
    ("Laptop Standard", "IT", 620, 0.24, 1.01),
    ("Tablet Enterprise", "IT", 410, 0.27, 1.12),
    ("Medical Sensor", "Medical", 180, 0.45, 1.15),
    ("Monitoring Device", "Medical ", 760, 0.39, 1.10),
    ("Industrial Scanner", "Accessories", 1450, 0.34, 1.03),
    ("Connectivity Module", "Accessories", 85, 0.31, 1.18),
    ("Legacy Workstation", "Services", 780, 0.18, 0.88),
    ("Accessory Kit", "Accessories", 55, 0.22, 1.05),
    ("Service Contract", "Services", 240, 0.62, 1.16)]

launch_years = np.array([2018, 2019, 2020, 2021, 2022, 2023, 2024])
launch_probabilities = np.array([0.08, 0.10, 0.15, 0.22, 0.18, 0.17, 0.10])

product_rows = []
product_id = 1000

for group, family, base_price, base_margin, growth_factor in product_groups:
    for product_number in range(1, PRODUCTS_PER_GROUP + 1):
        launch_year = int(rng.choice(launch_years, p=launch_probabilities))

        if "Legacy" in group:
            lifecycle = rng.choice(["Mature", "Decline"], p=[0.25, 0.75])
        elif launch_year >= 2024:
            lifecycle = rng.choice(
                ["New", "Growth", "Mature"], p=[0.65, 0.30, 0.05])
        elif launch_year >= 2022:
            lifecycle = rng.choice(
                ["New", "Growth", "Mature", "Decline"],
                p=[0.05, 0.45, 0.45, 0.05])
        else:
            lifecycle = rng.choice(
                ["Growth", "Mature", "Decline"], p=[0.15, 0.55, 0.30])

        price = base_price * rng.lognormal(0, 0.16)
        simulated_margin = np.clip(
            base_margin + rng.normal(0, 0.05), 0.08, 0.75)
        unit_cost = price * (1 - simulated_margin)

        product_rows.append(
            [
                product_id,
                f"SKU-{product_id}",
                family,
                group,
                f"{group} Model {product_number:02d}",
                launch_year,
                lifecycle,
                round(price, 2),
                round(unit_cost, 2),
                round(base_margin, 3),
                growth_factor])
        product_id += 1

dimProduct = pd.DataFrame(
    product_rows,
    columns=[
        "product_id",
        "sku",
        "product_family",
        "product_group",
        "product_name",
        "launch_year",
        "lifecycle_stage",
        "base_list_price_eur",
        "base_unit_cost_eur",
        "target_margin_pct",
        "product_growth_factor"])

In [36]:
print(f"dimProduct shape: {dimProduct.shape}")
display(dimProduct.sample(5, random_state=SEED))

lifecycle_check = (
    dimProduct.groupby(["launch_year", "lifecycle_stage"])
    .size()
    .unstack(fill_value=0))
display(lifecycle_check)

dimProduct shape: (200, 11)


,product_id,sku,product_family,product_group,product_name,launch_year,lifecycle_stage,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor
95,1095,SKU-1095,Medical,Monitoring Device,Monitoring Device Model 16,2021,Decline,871.67,495.03,0.39,1.10
15,1015,SKU-1015,IT,Laptop Pro,Laptop Pro Model 16,2022,Growth,725.75,527.43,0.29,1.08
30,1030,SKU-1030,IT,Laptop Standard,Laptop Standard Model 11,2022,Mature,526.35,395.31,0.24,1.01
158,1158,SKU-1158,Services,Legacy Workstation,Legacy Workstation Model 19,2023,Decline,874.37,706.42,0.18,0.88
128,1128,SKU-1128,Accessories,Connectivity Module,Connectivity Module Model 09,2023,Mature,105.73,68.22,0.31,1.18


lifecycle_stage,Decline,Growth,Mature,New
launch_year,,,,
2018,3,3,7,0
2019,14,2,10,0
2020,10,1,18,0
2021,16,7,19,0
2022,5,15,18,1
2023,5,15,16,0
2024,0,4,0,11


## 2.4 Customer dimension
Customer segment affects order size, discount behavior and churn risk. Larger customers receive a higher sampling weight later so enterprise and distributor accounts naturally contribute more transactions.

In [37]:
customer_segments = [
    "Enterprise",
    "Mid-Market",
    "SMB",
    "Public Sector",
    "Distributor"]
segment_scale = {
    "Enterprise": 8.0,
    "Mid-Market": 3.5,
    "SMB": 1.0,
    "Public Sector": 2.4,
    "Distributor": 5.5}
churn_risk = {
    "Enterprise": 0.03,
    "Mid-Market": 0.07,
    "SMB": 0.16,
    "Public Sector": 0.05,
    "Distributor": 0.09}
region_probabilities = np.array(
    [0.18, 0.11, 0.10, 0.09, 0.10, 0.16, 0.06, 0.08, 0.07, 0.05])

customer_rows = []

for customer_id in range(1, N_CUSTOMERS + 1):
    segment = rng.choice(
        customer_segments, p=[0.12, 0.28, 0.38, 0.10, 0.12])
    region_id = int(
        rng.choice(dimRegion["region_id"], p=region_probabilities))
    customer_start = pd.Timestamp("2020-01-01") + pd.Timedelta(
        days=int(rng.integers(0, 1500)))
    industry = rng.choice(
        [
            "Healthcare",
            "Manufacturing",
            "Retail",
            "Technology",
            "Logistics",
            "Education",
            "Government"])
    size_score = rng.lognormal(
        mean=np.log(segment_scale[segment]), sigma=0.55)

    customer_rows.append(
        [
            customer_id,
            f"CUST-{customer_id:05d}",
            segment,
            industry,
            region_id,
            customer_start.strftime("%Y-%m-%d"),
            round(size_score, 3),
            round(
                churn_risk[segment] * rng.uniform(0.65, 1.45), 3)])

dimCustomer = pd.DataFrame(
    customer_rows,
    columns=[
        "customer_id",
        "customer_code",
        "customer_segment",
        "industry",
        "region_id",
        "customer_since",
        "customer_size_score",
        "base_churn_probability"])

In [38]:
print(f"dimCustomer shape: {dimCustomer.shape}")
display(dimCustomer.sample(5, random_state=SEED))

customer_mix = (
    dimCustomer.groupby("customer_segment")
    .agg(
        customers=("customer_id", "count"),
        average_size_score=("customer_size_score", "mean"),
        average_churn_risk=("base_churn_probability", "mean"))
    .sort_values("customers", ascending=False))
display(customer_mix)


dimCustomer shape: (1200, 8)


,customer_id,customer_code,customer_segment,industry,region_id,customer_since,customer_size_score,base_churn_probability
1178,1179,CUST-01179,Public Sector,Education,5,2023-02-15,3.32,0.04
865,866,CUST-00866,Mid-Market,Healthcare,6,2020-08-06,6.63,0.06
101,102,CUST-00102,Enterprise,Logistics,1,2023-11-04,9.77,0.02
439,440,CUST-00440,SMB,Healthcare,1,2021-05-12,1.01,0.14
58,59,CUST-00059,Public Sector,Manufacturing,8,2022-05-06,2.32,0.05


,customers,average_size_score,average_churn_risk
customer_segment,,,
SMB,442,1.15,0.17
Mid-Market,334,4.11,0.07
Distributor,154,6.29,0.09
Enterprise,140,9.48,0.03
Public Sector,130,2.71,0.05


## 2.5 Sales representative dimension

We distribute representatives across all regions before assigning their seniority and quota. This guarantees that every market has at least one valid representative.

In [39]:
rep_region_ids = np.resize(
    dimRegion["region_id"].to_numpy(), N_SALES_REPS)
rng.shuffle(rep_region_ids)

rep_rows = []

for rep_id, region_id in enumerate(rep_region_ids, start=1):
    seniority = rng.choice(
        ["Junior", "Professional", "Senior", "Key Account"],
        p=[0.18, 0.42, 0.28, 0.12])
    base_quota = {
        "Junior": 750_000,
        "Professional": 1_400_000,
        "Senior": 2_300_000,
        "Key Account": 4_000_000,
    }[seniority]

    rep_rows.append(
        [
            rep_id,
            f"REP-{rep_id:03d}",
            int(region_id),
            seniority,
            int(base_quota * rng.uniform(0.80, 1.25))])

dimSalesRep = pd.DataFrame(
    rep_rows,
    columns=[
        "sales_rep_id",
        "sales_rep_code",
        "region_id",
        "seniority",
        "annual_quota_eur"])

reps_by_region = (
    dimSalesRep.merge(
        dimRegion[["region_id", "country"]], on="region_id", how="left")
    .groupby("country")
    .size()
    .rename("sales_reps")
    .sort_values(ascending=False)
    .to_frame())

print(f"dimSalesRep shape: {dimSalesRep.shape}")
display(dimSalesRep.head())
display(reps_by_region)


dimSalesRep shape: (85, 5)


,sales_rep_id,sales_rep_code,region_id,seniority,annual_quota_eur
0,1,REP-001,7,Professional,1558343
1,2,REP-002,1,Professional,1420920
2,3,REP-003,8,Senior,2611443
3,4,REP-004,3,Senior,2861897
4,5,REP-005,10,Professional,1590667


,sales_reps
country,
France,9
Germany,9
Italy,9
Spain,9
United Kingdom,9
Belgium,8
Finland,8
Ireland,8
Poland,8


# 3. Generate the sales fact table
Sales transactions are the central dataset. We first sample customer and product IDs, then assign a valid transaction date.

The date distribution contains two business effects:
- stronger demand in October–December
- lower demand in July–August
- approximately 6% annual growth

Sampling dates only after both `customer_since` and the product launch date prevents impossible sales history.

In [40]:
calendar_month = date_series.dt.month.to_numpy()
calendar_year = date_series.dt.year.to_numpy()

seasonal_factor = np.where(
    np.isin(calendar_month, [10, 11, 12]),
    1.30,
    np.where(np.isin(calendar_month, [7, 8]), 0.82, 1.00))
annual_growth_factor = 1.06 ** (calendar_year - 2021)

date_weights = seasonal_factor * annual_growth_factor
date_weights = date_weights / date_weights.sum()

date_weight_check = pd.DataFrame(
    {
        "date": date_series,
        "sampling_weight": date_weights})
date_weight_check["year"] = date_weight_check["date"].dt.year
date_weight_check["month"] = date_weight_check["date"].dt.month

display(
    date_weight_check.groupby("year")["sampling_weight"]
    .sum()
    .rename("share_of_sample")
    .to_frame())

,share_of_sample
year,
2021,0.18
2022,0.19
2023,0.20
2024,0.21
2025,0.22


## 3.1 Sample valid customers, products, and dates
The next cell deliberately exposes the intermediate IDs and minimum valid date. If a sampled date is too early only that date is resampled. The selected customer and product remain unchanged.

In [41]:
product_weights = (
    dimProduct["product_growth_factor"].to_numpy()
    * np.where(
        dimProduct["lifecycle_stage"].eq("Decline"),
        0.55,
        np.where(
            dimProduct["lifecycle_stage"].eq("New"), 0.65, 1.00)))
product_weights = product_weights / product_weights.sum()

customer_weights = dimCustomer["customer_size_score"].to_numpy()
customer_weights = customer_weights / customer_weights.sum()

sales_customer_ids = rng.choice(
    dimCustomer["customer_id"].to_numpy(),
    size=N_SALES_ROWS,
    p=customer_weights)
sales_product_ids = rng.choice(
    dimProduct["product_id"].to_numpy(),
    size=N_SALES_ROWS,
    p=product_weights)

customer_start_lookup = pd.to_datetime(
    dimCustomer.set_index("customer_id")["customer_since"])
product_launch_lookup = pd.to_datetime(
    dimProduct.set_index("product_id")["launch_year"].astype(str)
    + "-01-01")

sampled_customer_starts = customer_start_lookup.loc[
    sales_customer_ids
].to_numpy(dtype="datetime64[ns]")
sampled_product_launches = product_launch_lookup.loc[
    sales_product_ids
].to_numpy(dtype="datetime64[ns]")

minimum_sales_dates = np.maximum(
    sampled_customer_starts, sampled_product_launches)
minimum_sales_dates = np.maximum(
    minimum_sales_dates, np.datetime64(DATE_START))

sales_dates = rng.choice(
    date_array, size=N_SALES_ROWS, p=date_weights)
invalid_date = sales_dates < minimum_sales_dates
resampling_rounds = 0

while invalid_date.any():
    sales_dates[invalid_date] = rng.choice(
        date_array, size=int(invalid_date.sum()), p=date_weights)
    invalid_date = sales_dates < minimum_sales_dates
    resampling_rounds += 1

    if resampling_rounds > 100:
        raise RuntimeError(
            "Could not generate valid sales dates. Check the configured date range.")

sampled_keys = pd.DataFrame(
    {
        "customer_id": sales_customer_ids[:5],
        "product_id": sales_product_ids[:5],
        "minimum_valid_date": minimum_sales_dates[:5],
        "sampled_sales_date": sales_dates[:5]})

print(f"Date-resampling rounds required: {resampling_rounds}")
print(f"Invalid dates remaining: {int(invalid_date.sum()):,}")
display(sampled_keys)

Date-resampling rounds required: 17
Invalid dates remaining: 0


,customer_id,product_id,minimum_valid_date,sampled_sales_date
0,267,1198,2022-01-01,2022-08-18
1,367,1109,2023-05-24,2024-09-01
2,636,1075,2021-01-01,2023-01-23
3,952,1055,2023-11-30,2025-10-28
4,557,1050,2023-01-01,2024-01-02


## 3.2 Enrich the sampled keys
Join descriptive attributes required to calculate commercial measures.

In [42]:
factSales = pd.DataFrame(
    {
        "sales_id": np.arange(1, N_SALES_ROWS + 1),
        "date": pd.to_datetime(sales_dates),
        "customer_id": sales_customer_ids,
        "product_id": sales_product_ids})

factSales = factSales.merge(
    dimCustomer[
        [
            "customer_id",
            "region_id",
            "customer_segment",
            "customer_size_score",
            "base_churn_probability"]],
    on="customer_id",
    how="left",
    validate="m:1")
factSales = factSales.merge(
    dimProduct[
        [
            "product_id",
            "product_group",
            "product_family",
            "base_list_price_eur",
            "base_unit_cost_eur",
            "lifecycle_stage",
            "launch_year",
            "product_growth_factor"]],
    on="product_id",
    how="left",
    validate="m:1")
factSales = factSales.merge(
    dimRegion[
        [
            "region_id",
            "currency",
            "fx_to_eur",
            "market_growth_factor",
            "margin_factor"]],
    on="region_id",
    how="left",
    validate="m:1")

print(f"Intermediate factSales shape: {factSales.shape}")
display(factSales.head(3))


Intermediate factSales shape: (120000, 19)


,sales_id,date,customer_id,product_id,region_id,customer_segment,customer_size_score,base_churn_probability,product_group,product_family,base_list_price_eur,base_unit_cost_eur,lifecycle_stage,launch_year,product_growth_factor,currency,fx_to_eur,market_growth_factor,margin_factor
0,1,2022-08-18,267,1198,9,Public Sector,2.53,0.07,Service Contract,Services,298.30,109.55,Mature,2022,1.16,EUR,1.00,1.08,0.96
1,2,2024-09-01,367,1109,6,Mid-Market,4.22,0.09,Industrial Scanner,Accessories,"1,648.95",946.43,Mature,2020,1.03,EUR,1.00,1.07,1.00
2,3,2023-01-23,636,1075,3,Enterprise,7.04,0.02,Medical Sensor,Medical,227.78,154.51,Mature,2021,1.15,EUR,1.00,1.09,0.94


## 3.3 Assign region-compatible sales representatives
A representative should normally manage transactions in their own region. We therefore sample from the subset of representatives belonging to each transaction's region.

In [43]:
rep_ids_by_region = {
    region_id: group["sales_rep_id"].to_numpy()
    for region_id, group in dimSalesRep.groupby("region_id")}

assigned_rep_ids = np.empty(len(factSales), dtype=int)

for region_id, row_indices in factSales.groupby("region_id").groups.items():
    row_indices = np.asarray(row_indices, dtype=int)
    assigned_rep_ids[row_indices] = rng.choice(
        rep_ids_by_region[int(region_id)], size=len(row_indices)    )

factSales["sales_rep_id"] = assigned_rep_ids

rep_region_check = factSales[
    ["sales_id", "region_id", "sales_rep_id"]
].merge(
    dimSalesRep[
        ["sales_rep_id", "region_id"]
    ].rename(columns={"region_id": "sales_rep_region_id"}),
    on="sales_rep_id",
    how="left",
    validate="m:1")

display(rep_region_check.head())
assert (
    rep_region_check["region_id"]
    == rep_region_check["sales_rep_region_id"]
).all()
print("Check passed: every assigned sales representative matches the transaction region.")


,sales_id,region_id,sales_rep_id,sales_rep_region_id
0,1,9,25,9
1,2,6,68,6
2,3,3,6,3
3,4,3,71,3
4,5,1,37,1


Check passed: every assigned sales representative matches the transaction region.


## 3.4 Calculate units, pricing, revenue, cost and margin
The commercial measures are:

Revenue = Units * Average Selling Price

Gross Profit = Revenue - Units * Unit Cost

Price inflation, lifecycle, customer segment, regional growth and random variation create realistic heterogeneity. A small percentage of unusually large orders is marked with `is_outlier_order`.

In [44]:
transaction_year = factSales["date"].dt.year.to_numpy()
price_inflation = 1 + 0.035 * (transaction_year - 2021)
cost_inflation = 1 + 0.025 * (transaction_year - 2021)

lifecycle_price_factor = np.select(
    [
        factSales["lifecycle_stage"].eq("New"),
        factSales["lifecycle_stage"].eq("Growth"),
        factSales["lifecycle_stage"].eq("Decline")],
    [1.12, 1.05, 0.93],
    default=1.00)

discount = np.select(
    [
        factSales["customer_segment"].eq("Enterprise"),
        factSales["customer_segment"].eq("Distributor"),
        factSales["customer_segment"].eq("SMB")],
    [
        rng.normal(0.14, 0.04, N_SALES_ROWS),
        rng.normal(0.20, 0.05, N_SALES_ROWS),
        rng.normal(0.05, 0.03, N_SALES_ROWS)],
    default=rng.normal(0.09, 0.04, N_SALES_ROWS))
discount = np.clip(discount, 0, 0.38)

units_base = np.select(
    [
        factSales["customer_segment"].eq("Enterprise"),
        factSales["customer_segment"].eq("Distributor"),
        factSales["customer_segment"].eq("Mid-Market"),
        factSales["customer_segment"].eq("SMB")],
    [
        rng.poisson(45, N_SALES_ROWS),
        rng.poisson(70, N_SALES_ROWS),
        rng.poisson(18, N_SALES_ROWS),
        rng.poisson(6, N_SALES_ROWS)],
    default=rng.poisson(12, N_SALES_ROWS))

units = (
    units_base
    * factSales["market_growth_factor"].to_numpy()
    * rng.lognormal(0, 0.35, N_SALES_ROWS)).astype(int)
units = np.maximum(1, units)

outlier_mask = rng.random(N_SALES_ROWS) < 0.012
units[outlier_mask] *= rng.integers(4, 12, outlier_mask.sum())

asp_eur = (
    factSales["base_list_price_eur"].to_numpy()
    * price_inflation
    * lifecycle_price_factor
    * (1 - discount)
    * rng.normal(1, 0.04, N_SALES_ROWS))
asp_eur = np.maximum(5, asp_eur)

revenue_eur = units * asp_eur
unit_cost_eur = (
    factSales["base_unit_cost_eur"].to_numpy()
    * cost_inflation
    * rng.normal(1, 0.035, N_SALES_ROWS)
    / factSales["margin_factor"].to_numpy())
unit_cost_eur = np.maximum(1, unit_cost_eur)

gross_profit_eur = revenue_eur - units * unit_cost_eur
gross_margin_pct = gross_profit_eur / revenue_eur

factSales["date_id"] = (
    factSales["date"].dt.strftime("%Y%m%d").astype(int))
factSales["year_month"] = (
    factSales["date"].dt.to_period("M").astype(str))
factSales["order_id"] = (
    "ORD-" + factSales["sales_id"].astype(str).str.zfill(8))
factSales["units"] = units
factSales["asp_eur"] = np.round(asp_eur, 2)
factSales["discount_pct"] = np.round(discount, 3)
factSales["revenue_eur"] = np.round(revenue_eur, 2)
factSales["revenue_local_currency"] = np.round(
    factSales["revenue_eur"] / factSales["fx_to_eur"], 2)
factSales["unit_cost_eur"] = np.round(unit_cost_eur, 2)
factSales["gross_profit_eur"] = np.round(gross_profit_eur, 2)
factSales["gross_margin_pct"] = np.round(gross_margin_pct, 4)
factSales["is_outlier_order"] = outlier_mask


### Intentional data-quality imperfections
Real operational data is rarely perfect. To make the dataset useful for SQL cleaning and preprocessing exercises, a small controlled share of values is set to missing. The missingness is limited to non-key fields.

In [45]:
intentional_missing_rates = {
    "discount_pct": 0.008,
    "sales_rep_id": 0.006,
    "gross_margin_pct": 0.004}

for column, missing_rate in intentional_missing_rates.items():
    missing_mask = rng.random(N_SALES_ROWS) < missing_rate
    factSales.loc[missing_mask, column] = np.nan

factSales["date"] = factSales["date"].dt.strftime("%Y-%m-%d")

factSales = factSales[
    [
        "sales_id",
        "order_id",
        "date_id",
        "date",
        "year_month",
        "customer_id",
        "product_id",
        "region_id",
        "sales_rep_id",
        "units",
        "asp_eur",
        "discount_pct",
        "revenue_eur",
        "revenue_local_currency",
        "currency",
        "unit_cost_eur",
        "gross_profit_eur",
        "gross_margin_pct",
        "is_outlier_order"]]


In [46]:
print(f"factSales shape: {factSales.shape}")
display(factSales.sample(5, random_state=SEED))

sales_kpis = pd.DataFrame(
    {
        "metric": [
            "Transactions",
            "Total units",
            "Revenue EUR",
            "Gross profit EUR",
            "Average gross margin",
            "Outlier orders"],
        "value": [
            len(factSales),
            factSales["units"].sum(),
            factSales["revenue_eur"].sum(),
            factSales["gross_profit_eur"].sum(),
            factSales["gross_margin_pct"].mean(),
            factSales["is_outlier_order"].sum()]})
display(sales_kpis)

missing_sales_fields = (
    factSales.isna()
    .mean()
    .mul(100)
    .loc[lambda values: values > 0]
    .rename("missing_pct")
    .to_frame())
display(missing_sales_fields)


factSales shape: (120000, 19)


,sales_id,order_id,date_id,date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order
71787,71788,ORD-00071788,20241114,2024-11-14,2024-11,1100,1005,7,43.00,4,"1,333.70",0.06,"5,334.79","23,194.74",PLN,"1,035.41","1,193.15",0.22,False
67218,67219,ORD-00067219,20240210,2024-02-10,2024-02,393,1041,7,43.00,9,355.29,0.10,"3,197.62","13,902.70",PLN,310.58,402.39,0.13,False
54066,54067,ORD-00054067,20230927,2023-09-27,2023-09,906,1180,9,9.00,24,214.50,0.15,"5,147.94","5,147.94",EUR,80.78,"3,209.29",0.62,False
7168,7169,ORD-00007169,20240909,2024-09-09,2024-09,14,1196,3,52.00,80,252.69,0.12,"20,215.52","20,215.52",EUR,103.36,"11,946.60",0.59,False
29618,29619,ORD-00029619,20250513,2025-05-13,2025-05,128,1069,2,36.00,44,179.09,0.17,"7,879.97","7,879.97",EUR,117.26,"2,720.45",0.35,False


,metric,value
0,Transactions,"120,000.00"
1,Total units,"4,992,905.00"
2,Revenue EUR,"2,456,121,193.24"
3,Gross profit EUR,"483,100,905.03"
4,Average gross margin,0.23
5,Outlier orders,"1,412.00"


,missing_pct
sales_rep_id,0.64
discount_pct,0.81
gross_margin_pct,0.42


In [47]:
# Check the time rules
sales_time_check = factSales[
    ["sales_id", "date", "customer_id", "product_id"]
].merge(
    dimCustomer[["customer_id", "customer_since"]],
    on="customer_id",
    how="left",
    validate="m:1",
).merge(
    dimProduct[["product_id", "launch_year"]],
    on="product_id",
    how="left",
    validate="m:1")

sales_time_check["date"] = pd.to_datetime(sales_time_check["date"])
sales_time_check["customer_since"] = pd.to_datetime(
    sales_time_check["customer_since"])
sales_time_check["launch_date"] = pd.to_datetime(
    sales_time_check["launch_year"].astype(str) + "-01-01")

sales_before_customer = (
    sales_time_check["date"] < sales_time_check["customer_since"])
sales_before_launch = (
    sales_time_check["date"] < sales_time_check["launch_date"])

time_rule_summary = pd.DataFrame(
    {
        "rule": [
            "Sales before customer start",
            "Sales before product launch",
        ],
        "violating_rows": [
            int(sales_before_customer.sum()),
            int(sales_before_launch.sum())]})
display(time_rule_summary)

assert not sales_before_customer.any()
assert not sales_before_launch.any()
print("Time-rule checks passed.")


,rule,violating_rows
0,Sales before customer start,0
1,Sales before product launch,0


Time-rule checks passed.


# 4. Create supporting fact tables
The remaining tables reuse the dimensions and the monthly sales aggregate. Each section includes an intermediate sample so its grain and measures can be inspected before export.

## 4.1 Monthly sales aggregate
Forecast and inventory tables operate at a monthly product-region level. We therefore aggregate the transaction-level sales once and reuse the result.

In [48]:
monthly_sales = (
    factSales.groupby(
        ["year_month", "product_id", "region_id"], dropna=False)
    .agg(
        actual_units=("units", "sum"),
        actual_revenue_eur=("revenue_eur", "sum"),
        actual_gp_eur=("gross_profit_eur", "sum"))
    .reset_index())

print(f"Monthly product-region combinations: {len(monthly_sales):,}")
display(monthly_sales.head())

Monthly product-region combinations: 60,199


,year_month,product_id,region_id,actual_units,actual_revenue_eur,actual_gp_eur
0,2021-01,1001,4,9,"7,462.38",571.77
1,2021-01,1001,6,17,"13,556.40","1,653.30"
2,2021-01,1001,8,23,"19,292.04","2,150.05"
3,2021-01,1001,9,13,"10,089.28",-33.19
4,2021-01,1001,10,38,"29,493.53",583.25


## 4.2 Forecast fact
Forecasts are noisy estimates of actual monthly performance. The simulated error allows forecast accuracy, bias and variance analyses in Power BI.

In [49]:
region_bias = dimRegion.set_index("region_id")[
    "market_growth_factor"
].to_dict()

forecast_work = monthly_sales.copy()
forecast_work["forecast_bias_factor"] = (
    forecast_work["region_id"].map(region_bias).fillna(1.0))
forecast_work["forecast_units"] = np.maximum(
    0,
    (
        forecast_work["actual_units"]
        * (1 + rng.normal(0, 0.13, len(forecast_work)))
        / forecast_work["forecast_bias_factor"])
    .round()
    .astype(int))
forecast_work["forecast_revenue_eur"] = np.round(
    forecast_work["actual_revenue_eur"]
    * (1 + rng.normal(0, 0.15, len(forecast_work))),
    2)
forecast_work["forecast_version"] = rng.choice(
    ["Budget", "Rolling Forecast", "Latest Estimate"],
    size=len(forecast_work),
    p=[0.25, 0.50, 0.25])

factForecast = forecast_work[
    [
        "year_month",
        "product_id",
        "region_id",
        "forecast_version",
        "forecast_units",
        "forecast_revenue_eur",
        "actual_units",
        "actual_revenue_eur"]
].copy()
factForecast.insert(
    0, "forecast_id", np.arange(1, len(factForecast) + 1))

print(f"factForecast shape: {factForecast.shape}")
display(factForecast.head())


factForecast shape: (60199, 9)


,forecast_id,year_month,product_id,region_id,forecast_version,forecast_units,forecast_revenue_eur,actual_units,actual_revenue_eur
0,1,2021-01,1001,4,Rolling Forecast,8,"7,148.15",9,"7,462.38"
1,2,2021-01,1001,6,Rolling Forecast,17,"14,959.58",17,"13,556.40"
2,3,2021-01,1001,8,Latest Estimate,20,"19,054.83",23,"19,292.04"
3,4,2021-01,1001,9,Rolling Forecast,13,"9,563.24",13,"10,089.28"
4,5,2021-01,1001,10,Rolling Forecast,36,"33,082.86",38,"29,493.53"


## 4.3 Inventory fact

Inventory balances follow a simple stock-flow relationship:

Ending Stock = max(0, Opening Stock + Production - Units Sold)

`stockout_flag` marks combinations where remaining stock is very low relative to demand.


In [50]:
inventory_work = monthly_sales[
    ["year_month", "product_id", "region_id", "actual_units"]].copy()
inventory_work["opening_stock_units"] = np.maximum(
    0,
    (
        inventory_work["actual_units"]
        * rng.uniform(0.60, 1.80, len(inventory_work))
    ).astype(int),
)
inventory_work["production_units"] = np.maximum(
    0,
    (
        inventory_work["actual_units"]
        * rng.uniform(0.75, 1.45, len(inventory_work))
    ).astype(int))
inventory_work["ending_stock_units"] = np.maximum(
    0,
    inventory_work["opening_stock_units"]
    + inventory_work["production_units"]
    - inventory_work["actual_units"])
inventory_work["stockout_flag"] = (
    inventory_work["ending_stock_units"]
    < inventory_work["actual_units"] * 0.12)
inventory_work["inventory_value_eur"] = np.round(
    inventory_work["ending_stock_units"]
    * rng.uniform(35, 650, len(inventory_work)),
    2)

factInventory = inventory_work.drop(columns=["actual_units"])
factInventory.insert(
    0, "inventory_id", np.arange(1, len(factInventory) + 1))

print(f"factInventory shape: {factInventory.shape}")
display(factInventory.head())
print(
    f"Stockout rate: {factInventory['stockout_flag'].mean():.1%}")


factInventory shape: (60199, 9)


,inventory_id,year_month,product_id,region_id,opening_stock_units,production_units,ending_stock_units,stockout_flag,inventory_value_eur
0,1,2021-01,1001,4,6,8,5,False,"2,053.73"
1,2,2021-01,1001,6,22,14,19,False,"8,085.90"
2,3,2021-01,1001,8,36,25,38,False,"10,735.77"
3,4,2021-01,1001,9,17,11,15,False,"2,239.53"
4,5,2021-01,1001,10,56,35,53,False,"20,336.82"


Stockout rate: 0.2%


## 4.4 Product cost fact
Standard and actual unit costs are generated for every product-month combination. Cost inflation is lower than price inflation, while random operational variation separates actual cost from standard cost.

In [51]:
months = pd.period_range(
    pd.Period(DATE_START, freq="M"),
    pd.Period(DATE_END, freq="M"),
    freq="M",
).astype(str)

cost_rows = []
cost_id = 1

for year_month in months:
    year = int(year_month[:4])

    for _, product in dimProduct[
        ["product_id", "base_unit_cost_eur"]
    ].iterrows():
        standard_cost = (
            product["base_unit_cost_eur"]
            * (1 + 0.025 * (year - 2021))
            * rng.normal(1, 0.025)        )
        actual_cost = standard_cost * rng.uniform(0.95, 1.08)

        cost_rows.append(
            [
                cost_id,
                year_month,
                int(product["product_id"]),
                round(standard_cost, 2),
                round(actual_cost, 2)])
        cost_id += 1

factCosts = pd.DataFrame(
    cost_rows,
    columns=[
        "cost_id",
        "year_month",
        "product_id",
        "standard_unit_cost_eur",
        "actual_unit_cost_eur"])

print(f"factCosts shape: {factCosts.shape}")
display(factCosts.head())

factCosts shape: (12000, 5)


,cost_id,year_month,product_id,standard_unit_cost_eur,actual_unit_cost_eur
0,1,2021-01,1000,690.85,718.51
1,2,2021-01,1001,711.19,675.77
2,3,2021-01,1002,734.48,748.53
3,4,2021-01,1003,738.29,743.27
4,5,2021-01,1004,750.02,717.72


## 4.5 CRM activity fact

CRM activities represent calls, emails, demos, visits, reviews, and training sessions. Activity dates are constrained by `customer_since` and the assigned representative comes from the customer's region.

In [52]:
crm_customer_ids = rng.choice(
    dimCustomer["customer_id"].to_numpy(),
    size=N_CRM_ACTIVITIES,
    p=customer_weights)
crm_customer_starts = customer_start_lookup.loc[
    crm_customer_ids
].to_numpy(dtype="datetime64[ns]")
crm_minimum_dates = np.maximum(
    crm_customer_starts, np.datetime64(DATE_START))

crm_dates = rng.choice(date_array, size=N_CRM_ACTIVITIES)
invalid_crm_dates = crm_dates < crm_minimum_dates

while invalid_crm_dates.any():
    crm_dates[invalid_crm_dates] = rng.choice(
        date_array, size=int(invalid_crm_dates.sum()))
    invalid_crm_dates = crm_dates < crm_minimum_dates

factCRMActivities = pd.DataFrame(
    {
        "activity_id": np.arange(1, N_CRM_ACTIVITIES + 1),
        "date": pd.to_datetime(crm_dates),
        "customer_id": crm_customer_ids,
        "activity_type": rng.choice(
            [
                "Call",
                "Email",
                "Demo",
                "Visit",
                "Business Review",
                "Training",
            ],
            N_CRM_ACTIVITIES,
            p=[0.34, 0.32, 0.10, 0.12, 0.06, 0.06]),
        "activity_minutes": np.maximum(
            5, rng.normal(35, 20, N_CRM_ACTIVITIES).astype(int)),
        "sentiment_score": np.round(
            np.clip(
                rng.normal(0.08, 0.55, N_CRM_ACTIVITIES), -1, 1),3)})

factCRMActivities = factCRMActivities.merge(
    dimCustomer[["customer_id", "region_id"]],
    on="customer_id",
    how="left",
    validate="m:1")

crm_rep_ids = np.empty(len(factCRMActivities), dtype=int)
for region_id, row_indices in factCRMActivities.groupby(
    "region_id"
).groups.items():
    row_indices = np.asarray(row_indices, dtype=int)
    crm_rep_ids[row_indices] = rng.choice(
        rep_ids_by_region[int(region_id)], size=len(row_indices))

factCRMActivities["sales_rep_id"] = crm_rep_ids
factCRMActivities["date_id"] = (
    factCRMActivities["date"].dt.strftime("%Y%m%d").astype(int))
factCRMActivities["customer_health_score"] = np.round(
    np.clip(
        65
        + factCRMActivities["sentiment_score"] * 15
        + rng.normal(0, 8, N_CRM_ACTIVITIES),
        1,
        100),1)
factCRMActivities["date"] = factCRMActivities["date"].dt.strftime(
    "%Y-%m-%d")

factCRMActivities = factCRMActivities[
    [
        "activity_id",
        "date_id",
        "date",
        "customer_id",
        "sales_rep_id",
        "activity_type",
        "activity_minutes",
        "sentiment_score",
        "customer_health_score"]]

print(f"factCRMActivities shape: {factCRMActivities.shape}")
display(factCRMActivities.head())


factCRMActivities shape: (35000, 9)


,activity_id,date_id,date,customer_id,sales_rep_id,activity_type,activity_minutes,sentiment_score,customer_health_score
0,1,20240516,2024-05-16,786,2,Business Review,18,0.13,68.10
1,2,20241113,2024-11-13,874,22,Visit,48,-0.71,62.40
2,3,20230913,2023-09-13,769,76,Email,40,0.73,81.90
3,4,20241004,2024-10-04,372,62,Call,29,-0.53,61.30
4,5,20230821,2023-08-21,54,17,Demo,36,0.70,78.50


## 4.6 Returns fact

We sample approximately 3.5% of eligible sales transactions. Return quantities cannot exceed sold quantities and each return date follows the original sale while remaining inside the date dimension.

In [53]:
latest_returnable_sale = pd.Timestamp(DATE_END) - pd.Timedelta(days=5)
eligible_returns = factSales.loc[
    pd.to_datetime(factSales["date"]) <= latest_returnable_sale]

n_returns = min(
    int(round(N_SALES_ROWS * RETURN_RATE)), len(eligible_returns))
sampled_sales = eligible_returns.sample(
    n=n_returns, random_state=SEED, replace=False)

factReturns = sampled_sales[
    [
        "sales_id",
        "date_id",
        "date",
        "customer_id",
        "product_id",
        "region_id",
        "units",
        "revenue_eur"]
].copy()
factReturns.insert(
    0, "return_id", np.arange(1, len(factReturns) + 1))

original_sale_dates = pd.to_datetime(factReturns["date"])
maximum_delays = np.minimum(
    90,
    (
        pd.Timestamp(DATE_END) - original_sale_dates
    ).dt.days.to_numpy())
return_delays = np.array(
    [
        rng.integers(5, int(maximum_delay) + 1)
        for maximum_delay in maximum_delays])
return_dates = original_sale_dates + pd.to_timedelta(
    return_delays, unit="D")

factReturns["date"] = return_dates.dt.strftime("%Y-%m-%d")
factReturns["date_id"] = return_dates.dt.strftime("%Y%m%d").astype(int)
factReturns["return_units"] = np.maximum(
    1,
    (
        factReturns["units"]
        * rng.uniform(0.05, 0.50, len(factReturns))
    ).astype(int))
factReturns["return_units"] = np.minimum(
    factReturns["return_units"], factReturns["units"])
factReturns["return_value_eur"] = np.round(
    factReturns["revenue_eur"]
    * factReturns["return_units"]
    / factReturns["units"], 2)
factReturns["return_reason"] = rng.choice(
    [
        "Defect",
        "Wrong Configuration",
        "Customer Cancellation",
        "Shipping Damage",
        "Other"],
    len(factReturns),
    p=[0.35, 0.22, 0.18, 0.15, 0.10])
factReturns = factReturns.drop(columns=["units", "revenue_eur"])

print(f"factReturns shape: {factReturns.shape}")
display(factReturns.head())


factReturns shape: (4200, 10)


,return_id,sales_id,date_id,date,customer_id,product_id,region_id,return_units,return_value_eur,return_reason
41006,1,41007,20240303,2024-03-03,270,1016,8,2,"2,198.06",Wrong Configuration
9517,2,9518,20230325,2023-03-25,953,1054,3,6,"2,895.60",Defect
38982,3,38983,20250801,2025-08-01,763,1133,6,2,136.27,Wrong Configuration
3135,4,3136,20240322,2024-03-22,611,1128,6,8,902.81,Wrong Configuration
38280,5,38281,20230711,2023-07-11,202,1016,8,4,"4,118.99",Shipping Damage


## 4.7 Sales pipeline fact

Opportunities are linked to a customer, product group and region-compatible representative. Win probability is influenced by pipeline stage, which makes weighted pipeline value more interpretable.

In [54]:
pipeline_customer_ids = rng.choice(
    dimCustomer["customer_id"].to_numpy(),
    size=N_OPPORTUNITIES,
    p=customer_weights)
pipeline_customer_starts = customer_start_lookup.loc[
    pipeline_customer_ids
].to_numpy(dtype="datetime64[ns]")
pipeline_minimum_dates = np.maximum(
    pipeline_customer_starts, np.datetime64(DATE_START))

opportunity_dates = rng.choice(date_array, size=N_OPPORTUNITIES)
invalid_opportunity_dates = (
    opportunity_dates < pipeline_minimum_dates)

while invalid_opportunity_dates.any():
    opportunity_dates[invalid_opportunity_dates] = rng.choice(
        date_array, size=int(invalid_opportunity_dates.sum()))
    invalid_opportunity_dates = (
        opportunity_dates < pipeline_minimum_dates)

factPipeline = pd.DataFrame(
    {
        "opportunity_id": np.arange(1, N_OPPORTUNITIES + 1),
        "created_date": pd.to_datetime(opportunity_dates),
        "customer_id": pipeline_customer_ids,
        "product_group": rng.choice(
            [group[0] for group in product_groups],
            N_OPPORTUNITIES),
        "stage": rng.choice(
            [
                "Lead",
                "Qualified",
                "Proposal",
                "Negotiation",
                "Closed Won",
                "Closed Lost"],
            N_OPPORTUNITIES,
            p=[0.25, 0.22, 0.18, 0.13, 0.12, 0.10]),
        "expected_value_eur": np.round(
            rng.lognormal(10.2, 1.0, N_OPPORTUNITIES), 2)})
factPipeline = factPipeline.merge(
    dimCustomer[["customer_id", "region_id"]],
    on="customer_id",
    how="left",
    validate="m:1")

pipeline_rep_ids = np.empty(len(factPipeline), dtype=int)
for region_id, row_indices in factPipeline.groupby(
    "region_id"
).groups.items():
    row_indices = np.asarray(row_indices, dtype=int)
    pipeline_rep_ids[row_indices] = rng.choice(
        rep_ids_by_region[int(region_id)], size=len(row_indices))
factPipeline["sales_rep_id"] = pipeline_rep_ids

stage_probability = factPipeline["stage"].map(
    {
        "Lead": 0.10,
        "Qualified": 0.25,
        "Proposal": 0.45,
        "Negotiation": 0.65,
        "Closed Won": 1.00,
        "Closed Lost": 0.00})
factPipeline["win_probability"] = np.round(
    np.clip(
        stage_probability
        + rng.normal(0, 0.08, N_OPPORTUNITIES),
        0,
        1),3)
factPipeline["expected_close_date"] = (
    factPipeline["created_date"]
    + pd.to_timedelta(
        rng.integers(20, 220, N_OPPORTUNITIES), unit="D")
).dt.strftime("%Y-%m-%d")
factPipeline["created_date_id"] = (
    factPipeline["created_date"].dt.strftime("%Y%m%d").astype(int))
factPipeline["created_date"] = factPipeline[
    "created_date"].dt.strftime("%Y-%m-%d")
factPipeline["weighted_pipeline_eur"] = np.round(
    factPipeline["expected_value_eur"]
    * factPipeline["win_probability"], 2)

factPipeline = factPipeline[
    [
        "opportunity_id",
        "created_date",
        "customer_id",
        "product_group",
        "sales_rep_id",
        "stage",
        "expected_value_eur",
        "win_probability",
        "expected_close_date",
        "created_date_id",
        "weighted_pipeline_eur"]]

print(f"factPipeline shape: {factPipeline.shape}")
display(factPipeline.head())


factPipeline shape: (15000, 11)


,opportunity_id,created_date,customer_id,product_group,sales_rep_id,stage,expected_value_eur,win_probability,expected_close_date,created_date_id,weighted_pipeline_eur
0,1,2022-06-06,893,Connectivity Module,53,Proposal,"3,925.15",0.45,2022-10-20,20220606,"1,758.47"
1,2,2025-11-10,312,Tablet Enterprise,61,Closed Won,"26,014.60",0.99,2026-05-03,20251110,"25,832.50"
2,3,2022-11-05,1141,Accessory Kit,12,Qualified,"41,945.33",0.27,2023-01-13,20221105,"11,157.46"
3,4,2025-10-26,185,Laptop Pro,30,Negotiation,"120,169.35",0.71,2026-04-19,20251026,"85,800.92"
4,5,2025-02-14,882,Service Contract,31,Proposal,"119,940.39",0.41,2025-06-09,20250214,"48,695.80"


# 5. Assemble and validate the relational dataset

Before writing files, we collect all tables in one dictionary and run several quality gates. If a critical rule fails, the notebook stops with an assertion rather than exporting inconsistent data.

In [55]:
tables = {
    "dimDate": dimDate,
    "dimRegion": dimRegion,
    "dimProduct": dimProduct,
    "dimCustomer": dimCustomer,
    "dimSalesRep": dimSalesRep,
    "factSales": factSales,
    "factForecast": factForecast,
    "factInventory": factInventory,
    "factCosts": factCosts,
    "factCRMActivities": factCRMActivities,
    "factReturns": factReturns,
    "factPipeline": factPipeline}

table_summary = pd.DataFrame(
    [
        {
            "table": table_name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "missing_cells": int(dataframe.isna().sum().sum())}
        for table_name, dataframe in tables.items()])
display(table_summary)


,table,rows,columns,duplicate_rows,missing_cells
0,dimDate,1826,11,0,0
1,dimRegion,10,7,0,0
2,dimProduct,200,11,0,0
3,dimCustomer,1200,8,0,0
4,dimSalesRep,85,5,0,0
5,factSales,120000,19,0,2242
6,factForecast,60199,9,0,0
7,factInventory,60199,9,0,0
8,factCosts,12000,5,0,0
9,factCRMActivities,35000,9,0,0


## 5.1 Output-column contract

This contract protects downstream SQL and Power BI work. It verifies that every output contains the same ordered column names as the original notebook. If a future edit accidentally renames or removes a field, the assertion identifies the affected table immediately.

In [56]:
expected_columns = {
    "dimDate": [
        "date",
        "date_id",
        "year",
        "quarter",
        "month",
        "month_name",
        "year_month",
        "day_of_week",
        "week_of_year",
        "is_month_end",
        "is_quarter_end"],
    "dimRegion": [
        "region_id",
        "country",
        "region",
        "currency",
        "fx_to_eur",
        "market_growth_factor",
        "margin_factor"],
    "dimProduct": [
        "product_id",
        "sku",
        "product_family",
        "product_group",
        "product_name",
        "launch_year",
        "lifecycle_stage",
        "base_list_price_eur",
        "base_unit_cost_eur",
        "target_margin_pct",
        "product_growth_factor"],
    "dimCustomer": [
        "customer_id",
        "customer_code",
        "customer_segment",
        "industry",
        "region_id",
        "customer_since",
        "customer_size_score",
        "base_churn_probability"],
    "dimSalesRep": [
        "sales_rep_id",
        "sales_rep_code",
        "region_id",
        "seniority",
        "annual_quota_eur"],
    "factSales": [
        "sales_id",
        "order_id",
        "date_id",
        "date",
        "year_month",
        "customer_id",
        "product_id",
        "region_id",
        "sales_rep_id",
        "units",
        "asp_eur",
        "discount_pct",
        "revenue_eur",
        "revenue_local_currency",
        "currency",
        "unit_cost_eur",
        "gross_profit_eur",
        "gross_margin_pct",
        "is_outlier_order"],
    "factForecast": [
        "forecast_id",
        "year_month",
        "product_id",
        "region_id",
        "forecast_version",
        "forecast_units",
        "forecast_revenue_eur",
        "actual_units",
        "actual_revenue_eur"],
    "factInventory": [
        "inventory_id",
        "year_month",
        "product_id",
        "region_id",
        "opening_stock_units",
        "production_units",
        "ending_stock_units",
        "stockout_flag",
        "inventory_value_eur"],
    "factCosts": [
        "cost_id",
        "year_month",
        "product_id",
        "standard_unit_cost_eur",
        "actual_unit_cost_eur"],
    "factCRMActivities": [
        "activity_id",
        "date_id",
        "date",
        "customer_id",
        "sales_rep_id",
        "activity_type",
        "activity_minutes",
        "sentiment_score",
        "customer_health_score"],
    "factReturns": [
        "return_id",
        "sales_id",
        "date_id",
        "date",
        "customer_id",
        "product_id",
        "region_id",
        "return_units",
        "return_value_eur",
        "return_reason"],
    "factPipeline": [
        "opportunity_id",
        "created_date",
        "customer_id",
        "product_group",
        "sales_rep_id",
        "stage",
        "expected_value_eur",
        "win_probability",
        "expected_close_date",
        "created_date_id",
        "weighted_pipeline_eur"]}

schema_results = []

for table_name, required_columns in expected_columns.items():
    actual_columns = list(tables[table_name].columns)
    missing_columns = [
        column
        for column in required_columns
        if column not in actual_columns]
    extra_columns = [
        column
        for column in actual_columns
        if column not in required_columns]

    schema_results.append(
        {
            "table": table_name,
            "required_columns": len(required_columns),
            "actual_columns": len(actual_columns),
            "missing_columns": ", ".join(missing_columns) or "None",
            "extra_columns": ", ".join(extra_columns) or "None",
            "exact_order_match": actual_columns == required_columns})

    assert not missing_columns, (
        f"{table_name} is missing required columns: {missing_columns}")
    assert actual_columns == required_columns, (
        f"{table_name} column order differs from the output contract.")

display(pd.DataFrame(schema_results))
print("Schema contract passed for all tables.")


,table,required_columns,actual_columns,missing_columns,extra_columns,exact_order_match
0,dimDate,11,11,None,None,True
1,dimRegion,7,7,None,None,True
2,dimProduct,11,11,None,None,True
3,dimCustomer,8,8,None,None,True
4,dimSalesRep,5,5,None,None,True
5,factSales,19,19,None,None,True
6,factForecast,9,9,None,None,True
7,factInventory,9,9,None,None,True
8,factCosts,5,5,None,None,True
9,factCRMActivities,9,9,None,None,True


Schema contract passed for all tables.


## 5.2 Primary-key and grain checks

Primary keys must be present and unique. We also check the intended business grain of the monthly tables.

In [57]:
primary_keys = {
    "dimDate": "date_id",
    "dimRegion": "region_id",
    "dimProduct": "product_id",
    "dimCustomer": "customer_id",
    "dimSalesRep": "sales_rep_id",
    "factSales": "sales_id",
    "factForecast": "forecast_id",
    "factInventory": "inventory_id",
    "factCosts": "cost_id",
    "factCRMActivities": "activity_id",
    "factReturns": "return_id",
    "factPipeline": "opportunity_id"}

primary_key_results = []

for table_name, primary_key in primary_keys.items():
    dataframe = tables[table_name]
    is_complete = dataframe[primary_key].notna().all()
    is_unique = dataframe[primary_key].is_unique

    primary_key_results.append(
        {
            "table": table_name,
            "primary_key": primary_key,
            "complete": is_complete,
            "unique": is_unique})

    assert is_complete, f"{table_name}.{primary_key} contains nulls."
    assert is_unique, f"{table_name}.{primary_key} is not unique."

grain_checks = {
    "factForecast month-product-region": not factForecast.duplicated(
        ["year_month", "product_id", "region_id"]
    ).any(),
    "factInventory month-product-region": not factInventory.duplicated(
        ["year_month", "product_id", "region_id"]
    ).any(),
    "factCosts month-product": not factCosts.duplicated(
        ["year_month", "product_id"]
    ).any()}

assert all(grain_checks.values()), "A monthly fact-table grain is duplicated."

display(pd.DataFrame(primary_key_results))
display(
    pd.DataFrame(
        {
            "grain_check": grain_checks.keys(),
            "passed": grain_checks.values()}))
print("Primary-key and grain checks passed.")


,table,primary_key,complete,unique
0,dimDate,date_id,True,True
1,dimRegion,region_id,True,True
2,dimProduct,product_id,True,True
3,dimCustomer,customer_id,True,True
4,dimSalesRep,sales_rep_id,True,True
5,factSales,sales_id,True,True
6,factForecast,forecast_id,True,True
7,factInventory,inventory_id,True,True
8,factCosts,cost_id,True,True
9,factCRMActivities,activity_id,True,True


,grain_check,passed
0,factForecast month-product-region,True
1,factInventory month-product-region,True
2,factCosts month-product,True


Primary-key and grain checks passed.


## 5.3 Foreign-key and relationship checks

Foreign keys connect fact tables to dimensions. Missing `sales_rep_id` values in `factSales` are intentionally allowed, but every non-missing representative ID must exist.


In [58]:
relationship_checks = {
    "factSales.date_id -> dimDate": factSales["date_id"]
    .isin(dimDate["date_id"])
    .all(),
    "factSales.customer_id -> dimCustomer": factSales["customer_id"]
    .isin(dimCustomer["customer_id"])
    .all(),
    "factSales.product_id -> dimProduct": factSales["product_id"]
    .isin(dimProduct["product_id"])
    .all(),
    "factSales.region_id -> dimRegion": factSales["region_id"]
    .isin(dimRegion["region_id"])
    .all(),
    "factSales.sales_rep_id -> dimSalesRep": factSales["sales_rep_id"]
    .dropna()
    .astype(int)
    .isin(dimSalesRep["sales_rep_id"])
    .all(),
    "factForecast.product_id -> dimProduct": factForecast[
        "product_id"]
    .isin(dimProduct["product_id"])
    .all(),
    "factInventory.region_id -> dimRegion": factInventory[
        "region_id"]
    .isin(dimRegion["region_id"])
    .all(),
    "factCosts.product_id -> dimProduct": factCosts["product_id"]
    .isin(dimProduct["product_id"])
    .all(),
    "factCRMActivities.customer_id -> dimCustomer": factCRMActivities[
        "customer_id"]
    .isin(dimCustomer["customer_id"])
    .all(),
    "factReturns.sales_id -> factSales": factReturns["sales_id"]
    .isin(factSales["sales_id"])
    .all(),
    "factPipeline.customer_id -> dimCustomer": factPipeline[
        "customer_id"]
    .isin(dimCustomer["customer_id"])
    .all()}

relationship_results = pd.DataFrame(
    {
        "relationship": relationship_checks.keys(),
        "passed": relationship_checks.values()})
display(relationship_results)

assert all(
    relationship_checks.values()
), "At least one foreign-key relationship is invalid."
print("Foreign-key checks passed.")


,relationship,passed
0,factSales.date_id -> dimDate,True
1,factSales.customer_id -> dimCustomer,True
2,factSales.product_id -> dimProduct,True
3,factSales.region_id -> dimRegion,True
4,factSales.sales_rep_id -> dimSalesRep,True
5,factForecast.product_id -> dimProduct,True
6,factInventory.region_id -> dimRegion,True
7,factCosts.product_id -> dimProduct,True
8,factCRMActivities.customer_id -> dimCustomer,True
9,factReturns.sales_id -> factSales,True


Foreign-key checks passed.


## 5.4 Business-rule checks

These tests cover value ranges, regional assignment, financial consistency, return timing, and pipeline timing.


In [59]:
sales_rep_region_validation = factSales.loc[
    factSales["sales_rep_id"].notna(),
    ["region_id", "sales_rep_id"],
].copy()
sales_rep_region_validation["sales_rep_id"] = (
    sales_rep_region_validation["sales_rep_id"].astype(int))
sales_rep_region_validation = sales_rep_region_validation.merge(
    dimSalesRep[
        ["sales_rep_id", "region_id"]
    ].rename(columns={"region_id": "sales_rep_region_id"}),
    on="sales_rep_id",
    how="left",
    validate="m:1")

return_timing = factReturns[
    ["sales_id", "date", "return_units"]
].merge(
    factSales[["sales_id", "date", "units", "revenue_eur"]],
    on="sales_id",
    how="left",
    suffixes=("_return", "_sale"),
    validate="m:1")

revenue_rounding_error = (
    factSales["revenue_eur"]
    - factSales["units"] * factSales["asp_eur"]
).abs()
gross_profit_rounding_error = (
    factSales["gross_profit_eur"]
    - (
        factSales["revenue_eur"]
        - factSales["units"] * factSales["unit_cost_eur"])
).abs()

business_rule_checks = {
    "No sales before customer start": not sales_before_customer.any(),
    "No sales before product launch": not sales_before_launch.any(),
    "Sales representatives match sales region": (
        sales_rep_region_validation["region_id"]
        == sales_rep_region_validation["sales_rep_region_id"]
    ).all(),
    "Discounts are between 0% and 38%": factSales[
        "discount_pct"]
    .dropna()
    .between(0, 0.38)
    .all(),
    "CRM sentiment is between -1 and 1": factCRMActivities[
        "sentiment_score"]
    .between(-1, 1)
    .all(),
    "Pipeline probability is between 0 and 1": factPipeline[
        "win_probability"]
    .between(0, 1)
    .all(),
    "Return date is on or after sale date": (
        pd.to_datetime(return_timing["date_return"])
        >= pd.to_datetime(return_timing["date_sale"])
    ).all(),
    "Return units do not exceed sold units": (
        return_timing["return_units"] <= return_timing["units"]
    ).all(),
    "Expected close is after opportunity creation": (
        pd.to_datetime(factPipeline["expected_close_date"])
        > pd.to_datetime(factPipeline["created_date"])
    ).all(),
    "Revenue formula is consistent after rounding": (
        revenue_rounding_error
        <= np.maximum(1.00, factSales["revenue_eur"].abs() * 0.001)
    ).all(),
    "Gross-profit formula is consistent after rounding": (
        gross_profit_rounding_error
        <= np.maximum(1.00, factSales["revenue_eur"].abs() * 0.001)
    ).all()}

business_rule_results = pd.DataFrame(
    {
        "business_rule": business_rule_checks.keys(),
        "passed": business_rule_checks.values()})
display(business_rule_results)

assert all(
    business_rule_checks.values()
), "At least one business-rule check failed."
print("Business-rule checks passed.")


,business_rule,passed
0,No sales before customer start,True
1,No sales before product launch,True
2,Sales representatives match sales region,True
3,Discounts are between 0% and 38%,True
4,CRM sentiment is between -1 and 1,True
5,Pipeline probability is between 0 and 1,True
6,Return date is on or after sale date,True
7,Return units do not exceed sold units,True
8,Expected close is after opportunity creation,True
9,Revenue formula is consistent after rounding,True


Business-rule checks passed.


# 6. Export the CSV files

In [60]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_rows = []

for table_name, dataframe in tables.items():
    file_path = OUTPUT_DIR / f"{table_name}.csv"
    dataframe.to_csv(
        file_path,
        sep=";",
        decimal=",",
        index=False)

    export_rows.append(
        {
            "table": table_name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "file": str(file_path)})

export_summary = pd.DataFrame(export_rows)
display(export_summary)
print(f"Export completed: {len(export_summary)} CSV files.")


,table,rows,columns,file
0,dimDate,1826,11,..\data\full_data\generated_data_tables\dimDat...
1,dimRegion,10,7,..\data\full_data\generated_data_tables\dimReg...
2,dimProduct,200,11,..\data\full_data\generated_data_tables\dimPro...
3,dimCustomer,1200,8,..\data\full_data\generated_data_tables\dimCus...
4,dimSalesRep,85,5,..\data\full_data\generated_data_tables\dimSal...
5,factSales,120000,19,..\data\full_data\generated_data_tables\factSa...
6,factForecast,60199,9,..\data\full_data\generated_data_tables\factFo...
7,factInventory,60199,9,..\data\full_data\generated_data_tables\factIn...
8,factCosts,12000,5,..\data\full_data\generated_data_tables\factCo...
9,factCRMActivities,35000,9,..\data\full_data\generated_data_tables\factCR...


Export completed: 12 CSV files.


## 7. Export to SSMS

In [ ]:
# SQL Server / SSMS import configuration
from urllib.parse import quote_plus
from decimal import Decimal

from sqlalchemy import create_engine, text
from sqlalchemy.types import (
    BigInteger,
    Boolean,
    Date,
    Integer,
    Numeric,
    SmallInteger,
    String,
)

SQL_SERVER = os.getenv("SQL_SERVER", r"localhost")
SQL_DATABASE = "RevenueAnalytics"
SQL_SCHEMA = "dbo"
SQL_DRIVER = "ODBC Driver 18 for SQL Server"

SQL_IF_EXISTS = "replace"

odbc_connection_string = (
    f"DRIVER={{{SQL_DRIVER}}};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(odbc_connection_string),
    # More reliable for nullable numeric/integer columns with pyodbc.
    # fast_executemany=True can raise SQLSTATE 22018 when a batch
    # contains a mixture of NULL and numeric values.
    fast_executemany=False,
)

# Explicit SQL Server datatypes.
# This prevents monetary and percentage columns from being imported as text
# and avoids locale-related decimal problems.
sql_dtypes = {
    "dimDate": {
        "date": Date(),
        "date_id": Integer(),
        "year": SmallInteger(),
        "quarter": SmallInteger(),
        "month": SmallInteger(),
        "month_name": String(20),
        "year_month": String(7),
        "day_of_week": SmallInteger(),
        "week_of_year": SmallInteger(),
        "is_month_end": Boolean(),
        "is_quarter_end": Boolean(),
    },
    "dimRegion": {
        "region_id": Integer(),
        "country": String(50),
        "region": String(50),
        "currency": String(3),
        "fx_to_eur": Numeric(18, 6),
        "market_growth_factor": Numeric(18, 6),
        "margin_factor": Numeric(18, 6),
    },
    "dimProduct": {
        "product_id": Integer(),
        "sku": String(50),
        "product_family": String(100),
        "product_group": String(100),
        "product_name": String(150),
        "launch_year": SmallInteger(),
        "lifecycle_stage": String(30),
        "base_list_price_eur": Numeric(18, 2),
        "base_unit_cost_eur": Numeric(18, 2),
        "target_margin_pct": Numeric(9, 6),
        "product_growth_factor": Numeric(9, 6),
    },
    "dimCustomer": {
        "customer_id": Integer(),
        "customer_code": String(50),
        "customer_segment": String(50),
        "industry": String(50),
        "region_id": Integer(),
        "customer_since": Date(),
        "customer_size_score": Numeric(18, 6),
        "base_churn_probability": Numeric(9, 6),
    },
    "dimSalesRep": {
        "sales_rep_id": Integer(),
        "sales_rep_code": String(50),
        "region_id": Integer(),
        "seniority": String(50),
        "annual_quota_eur": Numeric(18, 2),
    },
    "factSales": {
        "sales_id": BigInteger(),
        "order_id": String(50),
        "date_id": Integer(),
        "date": Date(),
        "year_month": String(7),
        "customer_id": Integer(),
        "product_id": Integer(),
        "region_id": Integer(),
        "sales_rep_id": Integer(),
        "units": Integer(),
        "asp_eur": Numeric(18, 4),
        "discount_pct": Numeric(9, 6),
        "revenue_eur": Numeric(18, 2),
        "revenue_local_currency": Numeric(18, 2),
        "currency": String(3),
        "unit_cost_eur": Numeric(18, 4),
        "gross_profit_eur": Numeric(18, 2),
        "gross_margin_pct": Numeric(9, 6),
        "is_outlier_order": Boolean(),
    },
    "factForecast": {
        "forecast_id": BigInteger(),
        "year_month": String(7),
        "product_id": Integer(),
        "region_id": Integer(),
        "forecast_version": String(50),
        "forecast_units": Integer(),
        "forecast_revenue_eur": Numeric(18, 2),
        "actual_units": Integer(),
        "actual_revenue_eur": Numeric(18, 2),
    },
    "factInventory": {
        "inventory_id": BigInteger(),
        "year_month": String(7),
        "product_id": Integer(),
        "region_id": Integer(),
        "opening_stock_units": Integer(),
        "production_units": Integer(),
        "ending_stock_units": Integer(),
        "stockout_flag": Boolean(),
        "inventory_value_eur": Numeric(18, 2),
    },
    "factCosts": {
        "cost_id": BigInteger(),
        "year_month": String(7),
        "product_id": Integer(),
        "standard_unit_cost_eur": Numeric(18, 4),
        "actual_unit_cost_eur": Numeric(18, 4),
    },
    "factCRMActivities": {
        "activity_id": BigInteger(),
        "date_id": Integer(),
        "date": Date(),
        "customer_id": Integer(),
        "sales_rep_id": Integer(),
        "activity_type": String(50),
        "activity_minutes": Integer(),
        "sentiment_score": Numeric(9, 6),
        "customer_health_score": Numeric(9, 3),
    },
    "factReturns": {
        "return_id": BigInteger(),
        "sales_id": BigInteger(),
        "date_id": Integer(),
        "date": Date(),
        "customer_id": Integer(),
        "product_id": Integer(),
        "region_id": Integer(),
        "return_units": Integer(),
        "return_value_eur": Numeric(18, 2),
        "return_reason": String(100),
    },
    "factPipeline": {
        "opportunity_id": BigInteger(),
        "created_date": Date(),
        "customer_id": Integer(),
        "product_group": String(100),
        "sales_rep_id": Integer(),
        "stage": String(50),
        "expected_value_eur": Numeric(18, 2),
        "win_probability": Numeric(9, 6),
        "expected_close_date": Date(),
        "created_date_id": Integer(),
        "weighted_pipeline_eur": Numeric(18, 2),
    },
}

date_columns = {
    "dimDate": ["date"],
    "dimCustomer": ["customer_since"],
    "factSales": ["date"],
    "factCRMActivities": ["date"],
    "factReturns": ["date"],
    "factPipeline": ["created_date", "expected_close_date"],
}

boolean_columns = {
    "dimDate": ["is_month_end", "is_quarter_end"],
    "factSales": ["is_outlier_order"],
    "factInventory": ["stockout_flag"],
}

decimal_columns = {
    table_name: [
        column_name
        for column_name, sql_type in dtype_map.items()
        if isinstance(sql_type, Numeric)
    ]
    for table_name, dtype_map in sql_dtypes.items()
}

integer_columns = {
    table_name: [
        column_name
        for column_name, sql_type in dtype_map.items()
        if isinstance(sql_type, Integer)
        and not isinstance(sql_type, Boolean)
    ]
    for table_name, dtype_map in sql_dtypes.items()
}


def _python_int_or_none(value):
    """Return a native Python int or SQL NULL-compatible None."""
    if pd.isna(value):
        return None
    return int(value)


def _python_decimal_or_none(value):
    """Return Decimal or None and reject NaN/Infinity before SQL upload."""
    if pd.isna(value):
        return None

    decimal_value = Decimal(str(value))
    if not decimal_value.is_finite():
        raise ValueError(f"Non-finite numeric value encountered: {value}")

    return decimal_value


def _python_bool_or_none(value):
    """Return a native Python bool or None."""
    if pd.isna(value):
        return None
    return bool(value)


def prepare_csv_for_sql(table_name, file_path):
    """Read an exported CSV and coerce values to SQL-Server-safe Python types."""
    df = pd.read_csv(
        file_path,
        sep=";",
        decimal=",",
        encoding="utf-8",
    )

    # Dates -> Python date objects / None.
    for column in date_columns.get(table_name, []):
        if column in df.columns:
            parsed = pd.to_datetime(df[column], errors="raise")
            df[column] = pd.Series(
                [
                    None if pd.isna(value) else value.date()
                    for value in parsed
                ],
                index=df.index,
                dtype=object,
            )

    # Integer columns -> native Python int / None.
    # Important for nullable columns such as factSales.sales_rep_id:
    # pandas otherwise represents the complete column as float (25.0, 68.0, ...)
    # because it contains NaN values.
    for column in integer_columns.get(table_name, []):
        if column in df.columns:
            numeric = pd.to_numeric(df[column], errors="raise")

            non_null = numeric.dropna()
            if not ((non_null % 1) == 0).all():
                raise ValueError(
                    f"{table_name}.{column} contains non-integer values."
                )

            df[column] = pd.Series(
                [_python_int_or_none(value) for value in numeric],
                index=df.index,
                dtype=object,
            )

    # Booleans -> native Python bool / None.
    for column in boolean_columns.get(table_name, []):
        if column in df.columns:
            values = df[column]

            if pd.api.types.is_bool_dtype(values):
                converted = values
            else:
                normalized = values.astype(str).str.strip().str.lower()
                converted = normalized.map(
                    {
                        "true": True,
                        "false": False,
                        "1": True,
                        "0": False,
                        "nan": None,
                        "none": None,
                        "": None,
                    }
                )

                invalid_mask = converted.isna() & values.notna()
                if invalid_mask.any():
                    bad_values = values.loc[invalid_mask].unique()[:5]
                    raise ValueError(
                        f"{table_name}.{column} contains values that cannot "
                        f"be converted to BIT: {bad_values}"
                    )

            df[column] = pd.Series(
                [_python_bool_or_none(value) for value in converted],
                index=df.index,
                dtype=object,
            )

    # DECIMAL/NUMERIC columns -> Decimal / None.
    # This also converts intentional NaN values in discount_pct and
    # gross_margin_pct to SQL NULL instead of passing NaN to pyodbc.
    for column in decimal_columns.get(table_name, []):
        if column in df.columns:
            values = df[column]

            if pd.api.types.is_object_dtype(values):
                values = (
                    values.astype(str)
                    .str.strip()
                    .str.replace(",", ".", regex=False)
                    .replace(
                        {
                            "nan": None,
                            "None": None,
                            "": None,
                        }
                    )
                )

            numeric = pd.to_numeric(values, errors="coerce")

            # Distinguish genuine missing values from invalid strings.
            invalid_mask = numeric.isna() & pd.Series(values, index=df.index).notna()
            if invalid_mask.any():
                bad_values = pd.Series(values, index=df.index).loc[
                    invalid_mask
                ].unique()[:5]
                raise ValueError(
                    f"{table_name}.{column} contains invalid numeric values: "
                    f"{bad_values}"
                )

            df[column] = pd.Series(
                [_python_decimal_or_none(value) for value in numeric],
                index=df.index,
                dtype=object,
            )

    return df


# Confirm that the requested database can be reached before loading data.
with engine.connect() as connection:
    connected_database = connection.execute(
        text("SELECT DB_NAME()")
    ).scalar_one()

print(
    f"Connected to SQL Server '{SQL_SERVER}', "
    f"database '{connected_database}'."
)

sql_import_rows = []

# export_summary contains exactly the CSV files produced by this notebook run.
for _, export_row in export_summary.iterrows():
    table_name = export_row["table"]
    file_path = Path(export_row["file"])

    if table_name not in sql_dtypes:
        raise KeyError(
            f"No SQL datatype mapping defined for table '{table_name}'."
        )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Exported CSV not found: {file_path}"
        )

    sql_df = prepare_csv_for_sql(table_name, file_path)

    null_cells = int(sql_df.isna().sum().sum())

    print(
        f"Uploading {table_name}: "
        f"{len(sql_df):,} rows -> "
        f"{SQL_DATABASE}.{SQL_SCHEMA}.{table_name} "
        f"(NULL cells: {null_cells:,})"
    )

    sql_df.to_sql(
        name=table_name,
        con=engine,
        schema=SQL_SCHEMA,
        if_exists=SQL_IF_EXISTS,
        index=False,
        dtype=sql_dtypes[table_name],
        chunksize=1000,
    )

    with engine.connect() as connection:
        sql_row_count = connection.execute(
            text(
                f"SELECT COUNT_BIG(*) "
                f"FROM [{SQL_SCHEMA}].[{table_name}]"
            )
        ).scalar_one()

    if int(sql_row_count) != len(sql_df):
        raise AssertionError(
            f"Row-count mismatch for {table_name}: "
            f"CSV={len(sql_df):,}, SQL={int(sql_row_count):,}"
        )

    sql_import_rows.append(
        {
            "table": table_name,
            "csv_rows": len(sql_df),
            "sql_rows": int(sql_row_count),
            "database": SQL_DATABASE,
            "schema": SQL_SCHEMA,
            "status": "OK",
        }
    )

sql_import_summary = pd.DataFrame(sql_import_rows)
display(sql_import_summary) 

print(
    f"SQL import completed successfully: "
    f"{len(sql_import_summary)} tables loaded into "
    f"{SQL_DATABASE}.{SQL_SCHEMA}."
)
